# 01 Bronze Layer
**CSE 487 Phase 3, Team CTRL-Freaks** (Bronze layer by Josh Hu)

**Purpose:** Get the raw Buffalo 311 data into Databricks and save a copy of it as Delta tables, wihtout changing anything. Cleaning happns in `02_silver_layer`.

**Inputs:** teh same two City of buffalo files we used in Phase 1
- 311 Service Requests (July 2008 - May 2024): https://data.buffalony.gov/api/views/whkc-e5vr/rows.csv?accessType=DOWNLOAD
- 311 Service Requests (May 2024 - Present): https://data.buffalony.gov/api/views/3tj7-3tdz/rows.csv?accessType=DOWNLOAD

**Outputs:**
- `old311.csv` and `new311.csv` in the volume `/Volumes/workspace/ctrl_freaks/raw_files/`
- Delta tables `workspace.ctrl_freaks.bronze_311_old` and `workspace.ctrl_freaks.bronze_311_new`

**Assumptions:** we use Databricks Free Edition w the default `workspace` catalog. the data is public, so no login is needed.

**How to run:** Run all, top to bottom. the first run downloads the files, so it takes a few minutes.

**If the download cell errors out:** the city site sometimes drops the connection halfway, whcih leaves a half file in the volume. Delete it from the volume and rerun, or download the 2 CSVs on your laptop and upload them to the volume (Catalog > workspace > ctrl_freaks > raw_files > Upload) w teh same names.

## Setup
**What**: Make a schema so where our tables go and a volume which is where our raw files go.

**Why**: everything for the project stays in one place so the rest of the team can find the tables.

In [0]:
# where everything lives in databricks, if ur catalog isnt called workspace change it here
catalogName = "workspace"
schemaName = "ctrl_freaks"
volumeName = "raw_files"

# IF NOT EXISTS so it doesnt break when we rerun
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogName}.{schemaName}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogName}.{schemaName}.{volumeName}")

volumePath = f"/Volumes/{catalogName}/{schemaName}/{volumeName}"
print("raw files go here:", volumePath)

## Data Source
| | Old dataset | New dataset |
|---|---|---|
| Name | 311 Service Requests (July 2008 - May 2024) | 311 Service Requests (May 2024 - Present) |
| From | City of Buffalo Open Data | City of Buffalo Open Data |
| Page | [whkc-e5vr](https://data.buffalony.gov/Quality-of-Life/311-Service-Requests-July-2008-May-2024-/whkc-e5vr/about_data) | [3tj7-3tdz](https://data.buffalony.gov/Quality-of-Life/311-Service-Requests-May-2024-Present-/3tj7-3tdz/about_data) |
| Format | CSV | CSV |
| License | Public Domain | Public Domain |
| Updated? | No, it stopped in May 2024 | Yes, new rows every day |
| Size | 347.7 MB | 78.1 MB |
| Rows | 1,129,274 | 212,811 |
| Columns | 34 | 28 |
| Our snapshot | downloaded Nov 19, 2026 | downloaded Nov 19, 2026 (has requests up to that day) |

## Step 1: Download the Files
**What:** Download both csvs from the cities website into our volume

**Why:** the new dataset keeps growing so we only download once. That way Silver, Gold, and the models all use the same copy of the data

**If it fails** (Databricks might block internet): download both links in your browser, then go to Catalog → workspace → ctrl_freaks → raw_files → Upload to this volume. Name them `old311.csv` and `new311.csv` and rerun.

In [0]:
import os
import urllib.request
import shutil
from datetime import datetime
#gonna run split cus old system is 2008 -2024 but new sys is may 2024 to now
oldUrl = "https://data.buffalony.gov/api/views/whkc-e5vr/rows.csv?accessType=DOWNLOAD"
newUrl = "https://data.buffalony.gov/api/views/3tj7-3tdz/rows.csv?accessType=DOWNLOAD"

oldFile = volumePath + "/old311.csv"
newFile = volumePath + "/new311.csv"

# only download if we dont have it yet bc the new data changes evry day
# and i dont want our numbers moving around on us mid project
# download to /tmp first then copy it over, that way if the download dies halfway
# we dont end up w a half file in the volume that gets skipped next time
if not os.path.exists(oldFile):
    print("downloading old data... takes a min")
    urllib.request.urlretrieve(oldUrl, "/tmp/old311.csv")
    shutil.copy("/tmp/old311.csv", oldFile)

if not os.path.exists(newFile):
    print("downloading new data...")
    urllib.request.urlretrieve(newUrl, "/tmp/new311.csv")
    shutil.copy("/tmp/new311.csv", newFile)

#prints out size + date we got it need these for the readme and proj saids so ig
for f in [oldFile, newFile]:
    sizeMb = os.path.getsize(f) / 1024 / 1024
    gotIt = datetime.fromtimestamp(os.path.getmtime(f))
    print(f, "|", round(sizeMb, 1), "MB | downloaded", gotIt)

## Step 2: Read the Files with Spark
**What**: load both CSVs into Spark DataFrames.
**Why**: every colum is read as text so the values stay exaclty how the city wrote them. If Spark guessed the types, IDs like `00990977` would lose thier leading zeros

**How**: `header` uses the first row as column names. `multiLine` and `escape` are there bc some status descriptions have commas, quotes, and line breaks inside them.

In [0]:
# not turning on inferSchema on purpose, that way everything stays a string
oldDf = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(oldFile)
newDf = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(newFile)

print("old columns:", len(oldDf.columns))
print("new columns:", len(newDf.columns))

In [0]:
# should all say string
oldDf.printSchema()

In [0]:
display(oldDf.limit(5))

In [0]:
newDf.printSchema()

In [0]:
display(newDf.limit(5))

## Step 3: Column Names and Tracking Columns
**What:** take the spaces out of the colum names (`Open Date` → `OpenDate`) and add two columns of our own: `sourceFile` and `ingestedAt`.

**Why:** Delta tables dont allow spaces in column names. Only the names change, not the data. the two new columns arent from the city. They js record whcih file each row came from and when we loaded it.

In [0]:
from pyspark.sql import functions as F

# delta hates spaces in column names so i js remove them
# printing old -> new so we can still match them back to the city's names
for col in oldDf.columns:
    newName = col.replace(" ", "")
    if newName != col:
        print("old:", col, "->", newName)
    oldDf = oldDf.withColumnRenamed(col, newName)

for col in newDf.columns:
    newName = col.replace(" ", "")
    if newName != col:
        print("new:", col, "->", newName)
    newDf = newDf.withColumnRenamed(col, newName)

In [0]:
#our 2 columns, everything else is still the city's data
oldDf = oldDf.withColumn("sourceFile", F.lit("old311.csv")).withColumn("ingestedAt", F.current_timestamp())
newDf = newDf.withColumn("sourceFile", F.lit("new311.csv")).withColumn("ingestedAt", F.current_timestamp())

print(oldDf.columns)
print(newDf.columns)

## Step 4: Save as Delta Tables
**What:** Count the rows, then save each DataFrame as a Delta table.

**Why:** we count first so we can check after saving that no rows got lost. `overwrite` means rerunning replaces the table instead of adding the rows twice.

In [0]:
# counting b4 saving so we have somthing to compare to after
oldCount = oldDf.count()
newCount = newDf.count()
print("old rows:", oldCount)
print("new rows:", newCount)

In [0]:
oldTable = f"{catalogName}.{schemaName}.bronze_311_old"
newTable = f"{catalogName}.{schemaName}.bronze_311_new"

#overwriteSchema is js in case we change a column later, delta gets mad otherwise
oldDf.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(oldTable)
newDf.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(newTable)

print("saved", oldTable)
print("saved", newTable)

## Step 5: Read the Tables Back
What: read both tables back from Delta and check the row counts, the columns, and the table format.

Why: the assignment says a write command alone doesnt prove anything, so we load the saved tables and compare.

In [0]:
#reading from the saved tables now, not the dataframes from b4
oldBronze = spark.table(oldTable)
newBronze = spark.table(newTable)

oldBack = oldBronze.count()
newBack = newBronze.count()

if oldBack == oldCount:
    print("old: wrote", oldCount, "| read back", oldBack, "| PASS")
else:
    print("old: wrote", oldCount, "| read back", oldBack, "| FAIL")

if newBack == newCount:
    print("new: wrote", newCount, "| read back", newBack, "| PASS")
else:
    print("new: wrote", newCount, "| read back", newBack, "| FAIL")

print("old columns match:", oldBronze.columns == oldDf.columns)
print("new columns match:", newBronze.columns == newDf.columns)

In [0]:
oldBronze.printSchema()

In [0]:
newBronze.printSchema()

In [0]:
# format should say delta
display(spark.sql(f"DESCRIBE DETAIL {oldTable}").select("name", "format", "location", "numFiles", "sizeInBytes"))
display(spark.sql(f"DESCRIBE DETAIL {newTable}").select("name", "format", "location", "numFiles", "sizeInBytes"))

## Step 6: Basic Checks
**What:** Count missing values, `UNKNOWN` text, repeated case IDs, and dates that dont match the expected format.

**Why:** these show what Silver has to clean up. we dont remove or fix anythign here. every row stays in Bronze.

In [0]:
# the city writes "UNKNOWN" instead of leaving stuff blank, so isNull alone misses those
# counting both. this one is slow bc its a lot of columns
print("OLD DATA")
for col in oldBronze.columns:
    # skip our own 2 columns, comparng a timestamp to "UNKNOWN" errors out
    if col == "sourceFile" or col == "ingestedAt":
        continue
    missing = oldBronze.filter(F.col(col).isNull()).count()
    unknown = oldBronze.filter(F.col(col) == "UNKNOWN").count()
    print(col, "| missing:", missing, "| UNKNOWN:", unknown)

In [0]:
print("NEW DATA")
for col in newBronze.columns:
    if col == "sourceFile" or col == "ingestedAt":
        continue
    missing = newBronze.filter(F.col(col).isNull()).count()
    unknown = newBronze.filter(F.col(col) == "UNKNOWN").count()
    print(col, "| missing:", missing, "| UNKNOWN:", unknown)

In [0]:
# evry reqeust should have a case id
# total rows - unique ids = how many ids show up more than once
oldMissingIds = oldBronze.filter(F.col("CaseReference").isNull()).count()
oldRepeatIds = oldBack - oldBronze.select("CaseReference").distinct().count()
print("old: missing case ids:", oldMissingIds, "| repeated case ids:", oldRepeatIds)

newMissingIds = newBronze.filter(F.col("CaseNumber").isNull()).count()
newRepeatIds = newBack - newBronze.select("CaseNumber").distinct().count()
print("new: missing case ids:", newMissingIds, "| repeated case ids:", newRepeatIds)

**Date check:** Both files write dates like `04/22/2024 11:19:00 AM`. `try_to_timestamp` turns each one into a real date and gives null if it doesnt match. That way we can count bad dates insted of crashing or dropping rows. The parsed date is only used for these checks and isnt saved.

In [0]:
dateFormat = "MM/dd/yyyy hh:mm:ss a"

oldCheck = oldBronze.withColumn("openParsed", F.try_to_timestamp(F.col("OpenDate"), F.lit(dateFormat)))
newCheck = newBronze.withColumn("openParsed", F.try_to_timestamp(F.col("CreatedDate"), F.lit(dateFormat)))

# bad = theres a date but it didnt parse, mising = no date at all
oldBadDates = oldCheck.filter(F.col("OpenDate").isNotNull() & F.col("openParsed").isNull()).count()
oldNoDates = oldCheck.filter(F.col("OpenDate").isNull()).count()
newBadDates = newCheck.filter(F.col("CreatedDate").isNotNull() & F.col("openParsed").isNull()).count()
newNoDates = newCheck.filter(F.col("CreatedDate").isNull()).count()

print("old: bad dates:", oldBadDates, "| missing dates:", oldNoDates)
print("new: bad dates:", newBadDates, "| missing dates:", newNoDates)

In [0]:
# old should be about 2008 -> may 2024, new should be may 2024 -> now
display(oldCheck.select(F.min("openParsed").alias("firstDate"), F.max("openParsed").alias("lastDate")))
display(newCheck.select(F.min("openParsed").alias("firstDate"), F.max("openParsed").alias("lastDate")))

## Summary

In [0]:
print("table | rows | columns")
print(oldTable, "|", oldBack, "|", len(oldBronze.columns))
print(newTable, "|", newBack, "|", len(newBronze.columns))

**What Bronze keeps:**
- Every row from both files, including 2008 to 2019.  Filtering to 2020+ happns in Silver.
- every value as text, exactly how the city wrote it.
- the only changes are the column names wihtout spaces and our two columns, `sourceFile` and `ingestedAt`
- Bad dates, mising values, repeated IDs, and `UNKNOWN` text are counted above but not removd.
- these row counts are bigger than Phase 2's 528,079 bc that number was after filtering to 2020+ and cleaning.

**For Silver:**
- Read `workspace.ctrl_freaks.bronze_311_old` and `workspace.ctrl_freaks.bronze_311_new`.
- Column names dont have spaces anymore (`Open Date` → `OpenDate`, `Created Date` → `CreatedDate`).
- Nothing is cleaned or merged yet, so all of Phase 1's cleaning steps still need to happen.